## Modular Pipeline Integration Test

In [ ]:
# Standard imports
import pandas as pd
import numpy as np
import torch
import os
import json
import sys
from pathlib import Path
from torch.utils.data import DataLoader


# Add project root to path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(project_root)


# Pipeline imports
from src.pipeline.data_ingestion import load_csv
from src.pipeline.preprocessing import preprocess_data
from src.pipeline.feature_engineering import generate_embeddings
from src.pipeline.datasets import ProductDataset
from src.pipeline.model_training import train_model
from src.pipeline.evaluation import evaluate_model
from src.pipeline.inference import predict
from src.initialization import load_environment

# Utility imports
from src.utils.vector_db_utils import VectorDBManager
from src.utils.data_cleaning import clean_stock_code

# Model imports
from src.models.cnn_model import CNNModel

load_environment()

### Data Ingestion & Cleaning
Load raw data from CSV, drop duplicates, apply cleaning/ preprocessing

In [16]:
raw_df = load_csv('../src/data/dataset/dataset.csv')
raw_df = raw_df.drop_duplicates()

# Clean and preprocess
cleaned_df = preprocess_data(raw_df)

# For vector DB: deduplicate by stockcode
cleaned_df_vdb = cleaned_df.drop_duplicates(subset='StockCode', keep='last')

# For classification: remove rare classes
counts = cleaned_df['StockCode'].value_counts()
filtered_df = cleaned_df[cleaned_df['StockCode'].isin(counts[counts > 1].index)]

filtered_df.head()

c:\Users\don\dev\ds_test\ds_task_1ab\src\utils\data_cleaning.py:45: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Quantity'] = df['Quantity'].fillna(1)
c:\Users\don\dev\ds_test\ds_task_1ab\src\utils\data_cleaning.py:46: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['InvoiceDate'] = df['InvoiceDate'].fillna('1970-01-01 00:00:00')
c:\Users\don\dev\ds_test\ds_task_1ab\src\utils\data_cleaning.py:47: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,178500,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,178500,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,178500,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,178500,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,178500,United Kingdom


### Feature Engineering & Embedding Generation
Generate feature embeddings for the deduplicated product data (for vector DB)

In [5]:
api_key = os.getenv('PINECONE_API_KEY')

vector_db_manager = VectorDBManager(api_key=api_key)
vector_db_manager.initialize()

embeddings = generate_embeddings(cleaned_df_vdb, vector_db_manager)

embeddings_df = pd.DataFrame(np.array(embeddings))
embeddings_df.head()

INFO: Using existing Pinecone index


INFO:src.utils.vector_db_utils:Using existing Pinecone index
INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: cpu
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: all-MiniLM-L6-v2


INFO: Model loaded


INFO:src.utils.vector_db_utils:Model loaded


INFO: Starting fresh upload


INFO:src.utils.vector_db_utils:Starting fresh upload


Generating embeddings for 3993 rows (batched)...


Batches:   0%|          | 0/125 [00:00<?, ?it/s]

Embedding generation complete.


,0,1,2,3,4,5,6,7,8,9,...,374,375,376,377,378,379,380,381,382,383
0,0.044934,0.047733,-0.031713,0.023898,-0.113119,0.020636,0.043290,0.058105,-0.022056,-0.009515,...,-0.107558,0.056651,0.087509,0.049710,0.045068,0.034528,-0.022875,-0.075215,-0.071923,0.016290
1,0.025311,-0.040255,-0.003945,-0.086670,0.041828,-0.073092,0.042201,-0.109977,-0.026584,0.006564,...,0.015641,-0.033279,0.019974,-0.080150,-0.053132,0.086473,0.124811,-0.035266,-0.053563,0.012651
2,0.025311,-0.040255,-0.003945,-0.086670,0.041828,-0.073092,0.042201,-0.109977,-0.026584,0.006564,...,0.015641,-0.033279,0.019974,-0.080150,-0.053132,0.086473,0.124811,-0.035266,-0.053563,0.012651
3,0.006931,-0.003061,-0.006435,0.012643,0.045201,-0.025058,0.055391,-0.064356,-0.039714,-0.002497,...,-0.056848,0.014105,0.026482,-0.008407,0.032257,0.013544,0.025055,-0.103874,-0.052060,0.061078
4,-0.074473,0.044887,-0.013407,-0.087875,-0.098879,0.052754,0.027268,0.017332,-0.102509,-0.055326,...,-0.013475,0.038868,0.049434,0.011181,-0.042980,0.013912,-0.045290,-0.089344,-0.026007,0.092768


### Model Training
Training the model with scrapped data

In [6]:
# Load and clean the CNN data
cnn_df = pd.read_csv('../src/data/dataset/CNN_Model_Train_Data.csv')
ecommerce_df = pd.read_csv('../src/data/dataset/cleaned_ecommerce_data.csv')
cnn_df = clean_stock_code(cnn_df)

In [7]:
# Merge product descriptions (optional, for richer metadata)
product_descriptions = (ecommerce_df[['StockCode', 'Description']]
                        .drop_duplicates()
                        .dropna(subset=['Description']))
merged_df = pd.merge(cnn_df, product_descriptions, on='StockCode', how='left')
merged_df

,StockCode,Description
0,22384,LUNCH BAG PINK POLKADOT
1,22727,ALARM CLOCK BAKELIKE RED
2,22112,CHOCOLATE HOT WATER BOTTLE
3,23298,SPOTTY BUNTING
4,20726,LUNCH BAG WOODLAND
5,21034,REX CASH+CARRY JUMBO SHOPPER
6,21931,JUMBO STORAGE BAG SUKI
7,22139,RETROSPOT TEA SET CERAMIC 11 PC
8,22077,6 RIBBONS RUSTIC CHARM
9,22423,REGENCY CAKESTAND 3 TIER


In [8]:
static_dir = Path('../static/images')
static_dir.mkdir(parents=True, exist_ok=True)
for stock_code in merged_df['StockCode'].unique():
    product_dir = static_dir / stock_code
    product_dir.mkdir(exist_ok=True)

In [9]:
# Run the training script
from src.scripts.train_cnn_from_scratch import main

try:
    history = main(
        project_root_str=project_root,
        batch_size=32,
        num_epochs=50,
        learning_rate=0.001
    )
    print("\nTraining completed successfully!")
    print(f"Best validation accuracy: {history['best_val_acc']:.2f}%")
except Exception as e:
    print(f"Error occurred: {str(e)}")
    raise

INFO: Loaded 506 images for training


INFO:src.scripts.train_cnn_from_scratch:Loaded 506 images for training


INFO: Class distribution:


INFO:src.scripts.train_cnn_from_scratch:Class distribution:


INFO: Class 6: 66 images


...

INFO: Training completed successfully


INFO:src.scripts.train_cnn_from_scratch:Training completed successfully


INFO: Saved StockCode-to-index mapping to c:\Users\don\dev\ds_test\ds_task_1ab\models\stockcode_to_index.json


INFO:src.scripts.train_cnn_from_scratch:Saved StockCode-to-index mapping to c:\Users\don\dev\ds_test\ds_task_1ab\models\stockcode_to_index.json



Training completed successfully!
Best validation accuracy: 0.00%


### Evaluation

In [10]:
images_root = r'C:\Users\don\dev\ds_test\ds_task_1ab\model_test'
image_paths = []
stockcodes = []

# Scan all folders and collect image paths
for stockcode in os.listdir(images_root):
    folder_path = os.path.join(images_root, stockcode)
    if os.path.isdir(folder_path):
        for fname in os.listdir(folder_path):
            if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                image_paths.append(os.path.join(folder_path, fname))
                stockcodes.append(stockcode)

# Build the DataFrame
df = pd.DataFrame({'image_path': image_paths, 'StockCode': stockcodes})
df

,image_path,StockCode
0,C:\Users\don\dev\ds_test\ds_task_1ab\model_tes...,20726
1,C:\Users\don\dev\ds_test\ds_task_1ab\model_tes...,21034
2,C:\Users\don\dev\ds_test\ds_task_1ab\model_tes...,21931
3,C:\Users\don\dev\ds_test\ds_task_1ab\model_tes...,22077
4,C:\Users\don\dev\ds_test\ds_task_1ab\model_tes...,22112
5,C:\Users\don\dev\ds_test\ds_task_1ab\model_tes...,22139
6,C:\Users\don\dev\ds_test\ds_task_1ab\model_tes...,22384
7,C:\Users\don\dev\ds_test\ds_task_1ab\model_tes...,22423
8,C:\Users\don\dev\ds_test\ds_task_1ab\model_tes...,22727
9,C:\Users\don\dev\ds_test\ds_task_1ab\model_tes...,23298


In [11]:
with open('C:/Users/don/dev/ds_test/ds_task_1ab/models/stockcode_to_index.json') as f:
    stockcode_to_index = json.load(f)

# Assign labels using the correct mapping
df['label'] = df['StockCode'].map(stockcode_to_index)
labels = df['label'].tolist()
df

,image_path,StockCode,label
0,C:\Users\don\dev\ds_test\ds_task_1ab\model_tes...,20726,4
1,C:\Users\don\dev\ds_test\ds_task_1ab\model_tes...,21034,5
2,C:\Users\don\dev\ds_test\ds_task_1ab\model_tes...,21931,6
3,C:\Users\don\dev\ds_test\ds_task_1ab\model_tes...,22077,8
4,C:\Users\don\dev\ds_test\ds_task_1ab\model_tes...,22112,2
5,C:\Users\don\dev\ds_test\ds_task_1ab\model_tes...,22139,7
6,C:\Users\don\dev\ds_test\ds_task_1ab\model_tes...,22384,0
7,C:\Users\don\dev\ds_test\ds_task_1ab\model_tes...,22423,9
8,C:\Users\don\dev\ds_test\ds_task_1ab\model_tes...,22727,1
9,C:\Users\don\dev\ds_test\ds_task_1ab\model_tes...,23298,3


In [12]:
# Prepare the dataset and DataLoader
image_paths = df['image_path']  # List of image paths for each row in train_df
labels = df['label']       # List of class indices for each row in train_df

train_dataset = ProductDataset(image_paths, labels, transform=None)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=False)

# Load the trained model
num_classes = len(set(labels))
model = CNNModel(num_classes)
model.load_state_dict(torch.load('C:/Users/don/dev/ds_test/ds_task_1ab/models/best_model.pth', map_location='cpu'))
model.eval()

# Evaluate
from src.pipeline.evaluation import evaluate_model
accuracy, metrics = evaluate_model(model, train_loader, labels)
print("Training set accuracy:", accuracy)
print("Metrics:", metrics)

Training set accuracy: 0.9
Metrics: {'accuracy': 0.9}


### Inference

In [13]:
from src.models.cnn_model import CNNModel
from torchvision import transforms
from PIL import Image
import torch


# Path to your trained model and label mapping
model_path = 'C:/Users/don/dev/ds_test/ds_task_1ab/models/best_model.pth'
num_classes = 10

# Load the model
model = CNNModel(num_classes)
model.load_state_dict(torch.load(model_path, map_location='cpu'))
model.eval()

# Define the same transforms as used in training
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load and preprocess the image
img_path = '../model_test/1.jpg'
image = Image.open(img_path).convert('RGB')
input_tensor = transform(image).unsqueeze(0)

# Run inference
with torch.no_grad():
    output = model(input_tensor)
    predicted_class_idx = output.argmax(dim=1).item()
    print("Predicted class index:", predicted_class_idx)

Predicted class index: 1


In [14]:
with open('C:/Users/don/dev/ds_test/ds_task_1ab/models/stockcode_to_index.json') as f:
    stockcode_to_index = json.load(f)
index_to_stockcode = {v: k for k, v in stockcode_to_index.items()}

"Predicted StockCode:", index_to_stockcode[predicted_class_idx]

('Predicted StockCode:', '22727')